# V2 Phase 11 — Colab GPU live artefact (`llama_cpp`)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

Validates the same live runner used by `app/streamlit_app.py` on a **fresh** question:
- Single-Agent RAG
- Multi-Agent RAG
- Uncertainty / Abstention RAG

Uses **llama_cpp + Qwen3-8B**, not mock. Does **not** run the 140-question benchmark. Does **not** start Phase 12.

## Setup

Push latest V2 (including Phase 11) to branch `cursor/empty-v2-workspace`, then run all cells.

Requires Phase 8 KB on Drive at `MyDrive/MSc-RAG/artifacts/knowledge_base/`.

**Outputs:** `results/config/phase11_smoke_test.json`, `phase11_live_smoke.json`

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'smoke_live_artefact.py').is_file():
    raise FileNotFoundError(f'Phase 11 script missing at {V2_ROOT}. Push Phase 11 to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Restore knowledge base from Drive (or rebuild)

Does **not** copy the Mac Chroma database. Reuses the Colab-built index from Phase 8 when available.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
restored = True

for rel in ('artifacts/knowledge_base/index', 'artifacts/knowledge_base/documents'):
    src = DRIVE_ROOT / rel
    dst = V2 / 'knowledge_base' / rel.split('/')[-1]
    if not src.is_dir():
        print('Missing on Drive:', src)
        restored = False
        break
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('restored', dst)

if not restored:
    print('Drive KB not found — falling back to build_index.py (Option B)...')
    !PYTHONPATH=. python scripts/build_index.py --distractors 50

## 4. Index preflight validation

In [ ]:
!PYTHONPATH=. python scripts/validate_kb_index.py

## 5. Phase 11 live comparison (`llama_cpp`, one fresh question)

Same `run_live_comparison()` used by Streamlit. Three architectures, independently, no chaining.

In [ ]:
!PYTHONPATH=. python scripts/smoke_live_artefact.py --backend llama_cpp --fresh-only

## 6. Check UI-equivalent fields

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase11_runtime_fingerprint.json')
smoke = Path('results/config/phase11_smoke_test.json')
detail = Path('results/config/phase11_live_smoke.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
print('detail:', detail.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', data.get('actual'))
    env = data.get('environment') or {}
    print('device:', env.get('device'))
    print('gpu:', (env.get('gpu') or {}).get('name'))
    print('backend:', (env.get('model_config') or {}).get('backend'))
if detail.is_file():
    detail_data = json.loads(detail.read_text())
    print('backend:', detail_data.get('backend'))
    for comparison in detail_data.get('comparisons', []):
        print('---')
        print('source:', comparison.get('question_source'), 'qid:', comparison.get('question_id'))
        print('question:', (comparison.get('question') or '')[:160])
        for architecture, case in (comparison.get('results') or {}).items():
            vr = case.get('verification_result') or {}
            print(
                architecture,
                'n_evidence=', len(case.get('retrieved_evidence') or []),
                'answer_len=', len(case.get('answer') or ''),
                'verify=', vr.get('verification_score'),
                'confidence=', case.get('confidence'),
                'threshold=', case.get('threshold'),
                'decision=', case.get('decision'),
                'error=', case.get('error'),
            )

## 7. Save Phase 11 Colab results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase11')
dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase11_runtime_fingerprint.json',
    'phase11_smoke_test.json',
    'phase11_live_smoke.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)
print('Done:', dest)